# 02 - Exploratory Data Analysis (EDA)

This notebook explores sensor patterns and activity-specific signal structure in the MHEALTH dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use("seaborn-v0_8-darkgrid")

data_path = Path("..") / "data" / "raw" / "mhealth_raw_data.csv"
df = pd.read_csv(data_path)

sensor_columns = [
    "alx", "aly", "alz",
    "glx", "gly", "glz",
    "arx", "ary", "arz",
    "grx", "gry", "grz",
]

activity_map = {
    0: "Null (no activity)",
    1: "Standing still",
    2: "Sitting and relaxing",
    3: "Lying down",
    4: "Walking",
    5: "Climbing stairs",
    6: "Waist bends forward",
    7: "Frontal elevation of arms",
    8: "Knees bending (crouching)",
    9: "Cycling",
    10: "Jogging",
    11: "Running",
    12: "Jump front & back",
}

df["ActivityName"] = df["Activity"].map(activity_map)

## Signal Patterns for Selected Activities

In [ ]:
def plot_sensor_window(activity_id, features, window_size=300):
    subset = df[df["Activity"] == activity_id].head(window_size)
    plt.figure(figsize=(14, 4))
    for feature in features:
        plt.plot(subset.index, subset[feature], label=feature, alpha=0.8)
    plt.title(f"{activity_map[activity_id]} - Sensor Signals")
    plt.xlabel("Sample index")
    plt.ylabel("Sensor value")
    plt.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

plot_sensor_window(4, ["alx", "aly", "alz"])
plot_sensor_window(11, ["grx", "gry", "grz"])

## Feature Distributions and Activity Separation

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x="ActivityName", y="alx")
plt.xticks(rotation=45, ha="right")
plt.title("alx Distribution by Activity")
plt.tight_layout()
plt.show()

## Correlation Heatmap for Sensor Features

In [ ]:
plt.figure(figsize=(12, 10))
corr = df[sensor_columns].corr()
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Correlation Matrix for Sensor Features")
plt.tight_layout()
plt.show()

## Activity Counts by Subject

In [ ]:
activity_subject = df.groupby(["subject", "ActivityName"]).size().unstack(fill_value=0)
activity_subject.head()

plt.figure(figsize=(14, 6))
sns.heatmap(activity_subject, cmap="Blues", cbar_kws={"label": "Count"})
plt.title("Activity Counts per Subject")
plt.ylabel("Subject")
plt.xlabel("Activity")
plt.tight_layout()
plt.show()